# OPTUNE: Bayesian-Optimized TabNet for Cancer Classification
## A Rigorous Multi-Dataset Nested Cross-Validation Benchmark

**Upgraded experimental framework** accompanying the manuscript revision of
*"Bayesian-Optimized TabNet for Cancer Classification: A Comprehensive Benchmarking Study"*.

This notebook replaces the v1 single-dataset holdout experiment with a methodologically
rigorous, publication-grade study. The ten key upgrades over v1:

| # | v1 weakness | v2 fix |
|---|-------------|--------|
| 1 | Single 80/10/10 holdout; paper claimed 3×10 CV | Repeated stratified **nested** cross-validation (2×5 outer folds, 3-fold inner tuning) — code and manuscript now match |
| 2 | Best-of-30-runs selected and reported on validation AUC (selection bias) | Hyperparameters tuned **inside** each outer training fold only; outer test folds never touched during tuning |
| 3 | Single saturated dataset (WDBC) | **Four public cancer datasets** spanning n=116–961, 4–30 features, 6–55% prevalence |
| 4 | Unequal search spaces (TabNet rich, baselines tiny grids) | **Equal HPO budget**: identical TPE sampler, trial count, and inner-CV protocol for all 8 models |
| 5 | SMOTE inflated training metrics; "negative generalization gap" artifact | **Cost-sensitive class weighting**; natural data distributions preserved |
| 6 | Wilcoxon on CV folds (violates independence) | **Nadeau–Bengio variance-corrected t-tests** + Holm correction + effect sizes + Friedman/Nemenyi across datasets |
| 7 | No modern baselines | **LightGBM + CatBoost + MLP** added under identical budget |
| 8 | No probability quality assessment | **Calibration** (Brier, ECE, reliability curves) + **decision-curve analysis** |
| 9 | Interpretability = one global importance bar chart | **Attribution stability across folds** + three-way agreement (attention vs SHAP vs permutation) + instance-level masks |
| 10 | No leakage control in preprocessing | All imputation/scaling **fit inside each training fold** |

*TabPFN was evaluated for inclusion but its pretrained weights could not be retrieved in the
execution environment; the framework supports it as an optional model.*


## 1. Environment and configuration

In [ ]:
pip install numpy pandas scipy scikit-learn torch optuna xgboost lightgbm catboost pytorch-tabnet

In [ ]:
pip install optuna

In [ ]:
pip install torch torchvision torchaudio

In [ ]:
import platform, sklearn, torch, optuna, xgboost, lightgbm, catboost, pytorch_tabnet, numpy, pandas, scipy
for m in [numpy, pandas, scipy, sklearn, torch, optuna, xgboost, lightgbm, catboost]:
    print(f"{m.__name__:15s} {m.__version__}")
print(f"python          {platform.python_version()}")

## 2. Datasets

Four public cancer classification datasets, chosen to span sample size, dimensionality,
class balance, and difficulty:

| Dataset | Source | n | Features | Positive class | Prevalence |
|---|---|---|---|---|---|
| **WDBC** | UCI / sklearn | 569 | 30 (FNA nuclear morphology) | malignant | 37.3% |
| **Coimbra** | UCI 451 | 116 | 9 (serum biomarkers) | breast cancer patient | 55.2% |
| **Mammographic Mass** | UCI 161 | 961 | 4 (age, shape, margin, density) | malignant | 46.3% |
| **Cervical (Risk Factors)** | UCI 383 | 858 | 30 (demographics, history, STDs) | positive biopsy | 6.4% |

Preprocessing decisions (documented for reproducibility):
- **Mammographic**: the BI-RADS column is an expert *assessment*, not a patient measurement — it is
  dropped to prevent target leakage. Missing values (`?`) are median-imputed inside each training fold.
- **Cervical**: the three co-recorded screening outcomes (Hinselmann, Schiller, Citology) are dropped
  (using them to predict biopsy would be leakage); two columns with >85% missingness are dropped.
- All imputation and standardization is fit on each outer-training fold only.

In [ ]:
import pandas as pd, numpy as np
rows = []
for name in ["wdbc", "coimbra", "mammographic", "cervical"]:
    d = pd.read_csv(f"data/{name}.csv")
    rows.append({"dataset": name, "n": len(d), "features": d.shape[1]-1,
                 "prevalence": round(d.target.mean(), 3),
                 "missing_cells": int(d.isna().sum().sum())})
pd.DataFrame(rows)

## 3. Methods

**Nested cross-validation.** The outer loop is stratified 5-fold CV repeated twice
(10 outer folds). Within each outer training fold, every model's hyperparameters are tuned by
Optuna TPE (15 trials, multivariate) scored by **mean 3-fold inner-CV ROC-AUC**. The winning
configuration is refit on the full outer-training fold and evaluated once on the outer test fold.
Test folds therefore never influence hyperparameter selection, model selection, or preprocessing —
the estimates below are unbiased estimates of the *whole pipeline's* generalization.

**Equal tuning budget.** All eight models receive the same sampler, the same number of trials, and
the same inner-validation protocol. Default-configuration versions of every model are also evaluated
on the same outer folds, isolating the causal benefit of Bayesian optimization (Section 5).

**Class imbalance.** Cost-sensitive weighting (class-balanced loss / inverse-frequency sampling for
TabNet) instead of SMOTE: synthetic oversampling distorts training distributions, inflates training
metrics, and produced the misleading "negative generalization gap" in v1.

**Statistics.** Paired model comparisons use the Nadeau–Bengio variance-corrected t-test
(accounts for the dependence between overlapping training folds in repeated CV), with Holm
correction over the model family and paired Cohen's d_z effect sizes. Cross-dataset behaviour is
summarized with the Friedman test and Nemenyi critical distance. Wilcoxon signed-rank p-values are
reported alongside for continuity with v1.

**TabNet training.** Early stopping (patience 15, max 100 epochs) on an internal 15% stratified
split of the training fold; StepLR schedule; inverse-frequency sample weighting; batch-size guard
against single-sample BatchNorm batches.

In [ ]:
"""
OPTUNE v2 — Methodologically rigorous benchmark of Bayesian-optimized TabNet
against classical and modern tabular baselines for cancer classification.

Key design principles (fixes over v1):
  1. Repeated stratified NESTED cross-validation: hyperparameters are tuned on
     inner folds only; outer test folds are never touched during tuning or
     model selection -> unbiased generalization estimates.
  2. Equal HPO budget (same sampler, same trial count, same inner-CV protocol)
     for every tuned model -> fair comparison.
  3. All preprocessing (imputation, scaling) is fit inside each outer training
     fold only -> no leakage.
  4. Class imbalance handled by cost-sensitive weighting (no synthetic
     oversampling), so training/validation distributions remain natural and
     train-vs-test gaps are interpretable.
  5. Out-of-fold probability predictions are collected for calibration and
     decision-curve analysis.
  6. Default-configuration models are evaluated on the same outer folds to
     quantify the benefit of Bayesian optimization (tuned vs default).
"""
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import torch

torch.set_num_threads(1)

import optuna
from optuna.samplers import TPESampler

optuna.logging.set_verbosity(optuna.logging.WARNING)

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# ---------------------------------------------------------------------------
# Experiment configuration
# ---------------------------------------------------------------------------
CONFIG = {
    "n_repeats": 2,          # outer CV repeats
    "n_outer_folds": 5,      # outer CV folds
    "n_inner_folds": 3,      # inner CV folds for HPO scoring
    "n_trials": 15,          # Optuna TPE trials per (model, outer fold)
    "random_seed": 42,
    "tabnet_max_epochs": 100,
    "tabnet_patience": 15,
    "tabnet_batch_size": 128,
}

TUNED_MODELS = [
    "TabNet",
    "LogisticRegression",
    "SVM",
    "RandomForest",
    "XGBoost",
    "LightGBM",
    "CatBoost",
    "MLP",
]


# ---------------------------------------------------------------------------
# Model factory: default configs, search spaces, builders
# ---------------------------------------------------------------------------
def suggest_params(model_name, trial):
    if model_name == "TabNet":
        nd = trial.suggest_categorical("n_d", [8, 16, 24, 32])
        return {
            "n_d": nd,
            "n_a": nd,  # tied, as recommended by Arik & Pfister
            "n_steps": trial.suggest_int("n_steps", 3, 6),
            "gamma": trial.suggest_float("gamma", 1.0, 2.0),
            "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
            "lr": trial.suggest_float("lr", 5e-3, 3e-2, log=True),
            # PyTorch BatchNorm momentum semantics (pytorch-tabnet default 0.02):
            # search a small-momentum region that CONTAINS the default. The v1
            # manuscript's 0.90-0.98 range reflects TensorFlow decay semantics
            # and destroys BN running statistics under PyTorch.
            "momentum": trial.suggest_float("momentum", 0.01, 0.4, log=True),
        }
    if model_name == "LogisticRegression":
        return {
            "C": trial.suggest_float("C", 1e-3, 1e2, log=True),
            "penalty": trial.suggest_categorical("penalty", ["l1", "l2"]),
        }
    if model_name == "SVM":
        return {
            "C": trial.suggest_float("C", 1e-2, 1e2, log=True),
            "gamma": trial.suggest_float("gamma", 1e-4, 1e0, log=True),
        }
    if model_name == "RandomForest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 16),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 8),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        }
    if model_name == "XGBoost":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
            "max_depth": trial.suggest_int("max_depth", 2, 8),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    if model_name == "LightGBM":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 7, 63),
            "min_child_samples": trial.suggest_int("min_child_samples", 5, 40),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
    if model_name == "CatBoost":
        return {
            "iterations": trial.suggest_int("iterations", 100, 600, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 1e-2, 0.3, log=True),
            "depth": trial.suggest_int("depth", 3, 8),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 30.0, log=True),
        }
    if model_name == "MLP":
        width = trial.suggest_categorical("width", [32, 64, 128])
        depth = trial.suggest_int("depth", 1, 3)
        return {
            "width": width,
            "depth": depth,
            "alpha": trial.suggest_float("alpha", 1e-6, 1e-1, log=True),
            "learning_rate_init": trial.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        }
    raise ValueError(model_name)


DEFAULT_PARAMS = {
    "TabNet": {"n_d": 8, "n_a": 8, "n_steps": 3, "gamma": 1.3, "lambda_sparse": 1e-3,
               "lr": 2e-2, "momentum": 0.02},
    "LogisticRegression": {"C": 1.0, "penalty": "l2"},
    "SVM": {"C": 1.0, "gamma": "scale"},
    "RandomForest": {"n_estimators": 100, "max_depth": None, "min_samples_leaf": 1,
                     "max_features": "sqrt"},
    "XGBoost": {"n_estimators": 100, "learning_rate": 0.3, "max_depth": 6,
                "subsample": 1.0, "colsample_bytree": 1.0, "min_child_weight": 1,
                "reg_lambda": 1.0},
    "LightGBM": {"n_estimators": 100, "learning_rate": 0.1, "num_leaves": 31,
                 "min_child_samples": 20, "subsample": 1.0, "colsample_bytree": 1.0,
                 "reg_lambda": 0.0},
    "CatBoost": {"iterations": 500, "learning_rate": 0.03, "depth": 6, "l2_leaf_reg": 3.0},
    "MLP": {"width": 100, "depth": 1, "alpha": 1e-4, "learning_rate_init": 1e-3},
}


def build_model(model_name, params, seed, pos_weight):
    """pos_weight = n_neg / n_pos on the current training data."""
    if model_name == "LogisticRegression":
        return LogisticRegression(
            C=params["C"], penalty=params["penalty"],
            solver="liblinear", max_iter=4000,
            class_weight="balanced", random_state=seed)
    if model_name == "SVM":
        return SVC(C=params["C"], gamma=params["gamma"], kernel="rbf",
                   probability=True, class_weight="balanced", random_state=seed)
    if model_name == "RandomForest":
        return RandomForestClassifier(
            n_estimators=params["n_estimators"], max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            max_features=params["max_features"],
            class_weight="balanced", random_state=seed, n_jobs=1)
    if model_name == "XGBoost":
        return XGBClassifier(
            **{k: params[k] for k in ["n_estimators", "learning_rate", "max_depth",
                                      "subsample", "colsample_bytree",
                                      "min_child_weight", "reg_lambda"]},
            scale_pos_weight=pos_weight, random_state=seed,
            eval_metric="logloss", verbosity=0, n_jobs=1)
    if model_name == "LightGBM":
        return LGBMClassifier(
            **{k: params[k] for k in ["n_estimators", "learning_rate", "num_leaves",
                                      "min_child_samples", "subsample",
                                      "colsample_bytree", "reg_lambda"]},
            scale_pos_weight=pos_weight, random_state=seed, verbosity=-1, n_jobs=1)
    if model_name == "CatBoost":
        return CatBoostClassifier(
            iterations=params["iterations"], learning_rate=params["learning_rate"],
            depth=params["depth"], l2_leaf_reg=params["l2_leaf_reg"],
            class_weights=[1.0, pos_weight], random_seed=seed,
            verbose=0, allow_writing_files=False, thread_count=1)
    if model_name == "MLP":
        hidden = tuple([params["width"]] * params["depth"])
        return MLPClassifier(
            hidden_layer_sizes=hidden, alpha=params["alpha"],
            learning_rate_init=params["learning_rate_init"],
            max_iter=600, early_stopping=True, n_iter_no_change=20,
            random_state=seed)
    raise ValueError(model_name)


def fit_predict_tabnet(params, Xtr, ytr, Xva, yva, Xte, seed, cfg=CONFIG):
    """Fit TabNet with early stopping on (Xva, yva); return probas on Xte and the model."""
    model = TabNetClassifier(
        n_d=params["n_d"], n_a=params.get("n_a", params["n_d"]),
        n_steps=params["n_steps"], gamma=params["gamma"],
        lambda_sparse=params["lambda_sparse"],
        optimizer_params={"lr": params["lr"]},
        scheduler_params={"step_size": 30, "gamma": 0.7},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        momentum=max(0.01, min(0.98, params.get("momentum", 0.02))),
        verbose=0, seed=seed, device_name="cpu")
    bs = min(cfg["tabnet_batch_size"], max(32, len(ytr) // 4))
    # BatchNorm cannot process a trailing batch of size 1: drop it if present
    drop_last = (len(ytr) % bs == 1)
    model.fit(
        Xtr, ytr,
        eval_set=[(Xva, yva)], eval_metric=["auc"],
        max_epochs=cfg["tabnet_max_epochs"], patience=cfg["tabnet_patience"],
        batch_size=bs,
        weights=1,  # inverse class-frequency sampling
        drop_last=drop_last)
    proba = model.predict_proba(Xte)[:, 1]
    return proba, model


def eval_model_on_split(model_name, params, Xtr, ytr, Xte, seed):
    """Train on (Xtr,ytr) and return P(y=1) on Xte plus the fitted model.
    For TabNet an internal early-stopping split is carved from the training data."""
    pos_weight = float((ytr == 0).sum() / max(1, (ytr == 1).sum()))
    if model_name == "TabNet":
        Xtr2, Xva, ytr2, yva = train_test_split(
            Xtr, ytr, test_size=0.15, stratify=ytr, random_state=seed)
        proba, model = fit_predict_tabnet(params, Xtr2, ytr2, Xva, yva, Xte, seed)
        return proba, model
    model = build_model(model_name, params, seed, pos_weight)
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]
    return proba, model


# ---------------------------------------------------------------------------
# Metrics
# ---------------------------------------------------------------------------
def compute_metrics(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sens = tp / (tp + fn) if (tp + fn) else np.nan
    spec = tn / (tn + fp) if (tn + fp) else np.nan
    out = {
        "roc_auc": roc_auc_score(y_true, proba) if len(np.unique(y_true)) > 1 else np.nan,
        "pr_auc": average_precision_score(y_true, proba),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "sensitivity": sens,
        "specificity": spec,
        "f1": f1_score(y_true, pred, zero_division=0),
        "mcc": matthews_corrcoef(y_true, pred),
        "brier": brier_score_loss(y_true, proba),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    return out


# ---------------------------------------------------------------------------
# Nested CV driver
# ---------------------------------------------------------------------------
def load_dataset(path):
    df = pd.read_csv(path)
    y = df["target"].values.astype(int)
    X = df.drop(columns=["target"])
    feature_names = X.columns.tolist()
    return X.values.astype(np.float64), y, feature_names


def preprocess_fold(X_tr_raw, X_te_raw):
    """Impute (median) + standardize, fit on training fold only."""
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(imputer.fit_transform(X_tr_raw)).astype(np.float32)
    Xte = scaler.transform(imputer.transform(X_te_raw)).astype(np.float32)
    return Xtr, Xte


def tune_on_inner(model_name, X_tr, y_tr, seed, cfg=CONFIG):
    """Equal-budget TPE tuning scored by mean inner-CV ROC-AUC."""
    inner = StratifiedKFold(n_splits=cfg["n_inner_folds"], shuffle=True, random_state=seed)
    splits = list(inner.split(X_tr, y_tr))

    def objective(trial):
        params = suggest_params(model_name, trial)
        aucs = []
        for i, (itr, iva) in enumerate(splits):
            try:
                proba, _ = eval_model_on_split(
                    model_name, params, X_tr[itr], y_tr[itr], X_tr[iva],
                    seed=seed + i)
                aucs.append(roc_auc_score(y_tr[iva], proba))
            except Exception:
                return 0.0
        return float(np.mean(aucs))

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=seed, multivariate=True))
    study.optimize(objective, n_trials=cfg["n_trials"], show_progress_bar=False)
    trials_df = [
        {"number": t.number, "value": t.value, "params": t.params}
        for t in study.trials if t.value is not None
    ]
    best = suggest_params(model_name, optuna.trial.FixedTrial(study.best_params))
    return best, study.best_value, trials_df


def run_benchmark(dataset_name, data_path, out_dir, models=TUNED_MODELS, cfg=CONFIG):
    os.makedirs(out_dir, exist_ok=True)
    X, y, feature_names = load_dataset(data_path)
    seed0 = cfg["random_seed"]

    fold_records = []       # per (model, fold) metrics
    oof_records = []        # out-of-fold predictions
    tuning_records = []     # best params + optuna trials per fold
    importance_records = [] # TabNet importances per fold
    perm_records = []       # TabNet permutation importance per fold
    shap_records = []       # XGBoost mean|SHAP| per fold
    mask_blob = None        # TabNet instance masks (one representative fold)

    log = open(os.path.join(out_dir, f"{dataset_name}.log"), "a")

    def logp(msg):
        stamp = time.strftime("%H:%M:%S")
        log.write(f"[{stamp}] {msg}\n")
        log.flush()

    logp(f"=== {dataset_name}: n={len(y)}, p={X.shape[1]}, pos_rate={y.mean():.3f} ===")

    fold_id = 0
    for rep in range(cfg["n_repeats"]):
        outer = StratifiedKFold(n_splits=cfg["n_outer_folds"], shuffle=True,
                                random_state=seed0 + rep)
        for k, (tr_idx, te_idx) in enumerate(outer.split(X, y)):
            Xtr, Xte = preprocess_fold(X[tr_idx], X[te_idx])
            ytr, yte = y[tr_idx], y[te_idx]
            fold_seed = seed0 + 1000 * rep + 10 * k

            for model_name in models:
                t0 = time.time()
                # --- tuned model ---
                best_params, inner_auc, trials = tune_on_inner(
                    model_name, Xtr, ytr, fold_seed, cfg)
                proba, fitted = eval_model_on_split(
                    model_name, best_params, Xtr, ytr, Xte, fold_seed)
                m = compute_metrics(yte, proba)
                m.update({"model": model_name, "dataset": dataset_name,
                          "repeat": rep, "fold": k, "fold_id": fold_id,
                          "variant": "tuned", "inner_auc": inner_auc,
                          "tune_seconds": time.time() - t0})
                fold_records.append(m)

                # --- default model (same folds, no tuning) ---
                proba_d, _ = eval_model_on_split(
                    model_name, DEFAULT_PARAMS[model_name], Xtr, ytr, Xte, fold_seed)
                md = compute_metrics(yte, proba_d)
                md.update({"model": model_name, "dataset": dataset_name,
                           "repeat": rep, "fold": k, "fold_id": fold_id,
                           "variant": "default", "inner_auc": np.nan,
                           "tune_seconds": 0.0})
                fold_records.append(md)

                for i, idx in enumerate(te_idx):
                    oof_records.append({
                        "dataset": dataset_name, "model": model_name,
                        "repeat": rep, "fold": k, "sample_idx": int(idx),
                        "y_true": int(yte[i]), "proba": float(proba[i]),
                        "proba_default": float(proba_d[i])})

                tuning_records.append({
                    "dataset": dataset_name, "model": model_name,
                    "repeat": rep, "fold": k,
                    "best_params": {kk: (vv if not isinstance(vv, np.generic) else vv.item())
                                    for kk, vv in best_params.items()},
                    "inner_auc": inner_auc, "trials": trials})

                if model_name == "TabNet":
                    imp = fitted.feature_importances_
                    importance_records.append({
                        "dataset": dataset_name, "repeat": rep, "fold": k,
                        "importances": [float(v) for v in imp],
                        "feature_names": feature_names})

                # --- interpretability artifacts (guarded: never breaks the run) ---
                try:
                    if model_name == "TabNet":
                        pi = permutation_importance(
                            fitted, Xte, yte, scoring="roc_auc",
                            n_repeats=5, random_state=fold_seed)
                        rec = {"repeat": rep, "fold": k}
                        rec.update({fn: float(v) for fn, v in
                                    zip(feature_names, pi.importances_mean)})
                        perm_records.append(rec)
                        if mask_blob is None:
                            expl, _masks = fitted.explain(Xte)
                            mask_blob = {
                                "masks": np.asarray(expl, dtype=float),
                                "feature_names": np.array(feature_names, dtype=object),
                                "y_true": yte.astype(int)}
                    elif model_name == "XGBoost":
                        import xgboost as _xgb
                        contribs = fitted.get_booster().predict(
                            _xgb.DMatrix(Xte), pred_contribs=True)
                        mean_abs = np.abs(contribs[:, :-1]).mean(axis=0)  # drop bias col
                        rec = {"repeat": rep, "fold": k}
                        rec.update({fn: float(v) for fn, v in
                                    zip(feature_names, mean_abs)})
                        shap_records.append(rec)
                except Exception as _e:
                    logp(f"interpretability skipped ({model_name}, rep{rep} fold{k}): {_e}")

                logp(f"rep{rep} fold{k} {model_name:18s} tunedAUC={m['roc_auc']:.4f} "
                     f"defaultAUC={md['roc_auc']:.4f} ({time.time()-t0:.0f}s)")
            fold_id += 1

    pd.DataFrame(fold_records).to_csv(
        os.path.join(out_dir, f"{dataset_name}_fold_metrics.csv"), index=False)
    pd.DataFrame(oof_records).to_csv(
        os.path.join(out_dir, f"{dataset_name}_oof_predictions.csv"), index=False)
    with open(os.path.join(out_dir, f"{dataset_name}_tuning.json"), "w") as f:
        json.dump(tuning_records, f)
    with open(os.path.join(out_dir, f"{dataset_name}_tabnet_importances.json"), "w") as f:
        json.dump(importance_records, f)
    if perm_records:
        pd.DataFrame(perm_records).to_csv(
            os.path.join(out_dir, f"{dataset_name}_tabnet_permutation.csv"), index=False)
    if shap_records:
        pd.DataFrame(shap_records).to_csv(
            os.path.join(out_dir, f"{dataset_name}_xgb_shap.csv"), index=False)
    if mask_blob is not None:
        np.savez(os.path.join(out_dir, f"{dataset_name}_instance_masks.npz"), **mask_blob)
    logp("DONE")
    log.close()
    return fold_records

print('framework loaded')

In [ ]:
"""
OPTUNE v2 — Statistical analysis of nested-CV benchmark results.

Implements:
  - Nadeau-Bengio variance-corrected paired t-tests (repeated k-fold CV),
    with Holm correction and paired effect sizes (Cohen's dz).
  - Friedman test + average ranks across datasets (Nemenyi critical distance).
  - Calibration: reliability data, Brier score, expected calibration error.
  - Decision-curve analysis (net benefit).
  - Operating points: sensitivity at fixed specificity (and vice versa).
"""
import json
import os

import numpy as np
import pandas as pd
from scipy import stats

MODELS = ["TabNet", "LogisticRegression", "SVM", "RandomForest",
          "XGBoost", "LightGBM", "CatBoost", "MLP"]
DATASETS = ["wdbc", "coimbra", "mammographic", "cervical"]
RES = os.path.join('.', "results")


def load_all_folds():
    dfs = []
    for d in DATASETS:
        p = os.path.join(RES, f"{d}_fold_metrics.csv")
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    return pd.concat(dfs, ignore_index=True)


def load_all_oof():
    dfs = []
    for d in DATASETS:
        p = os.path.join(RES, f"{d}_oof_predictions.csv")
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    return pd.concat(dfs, ignore_index=True)


# ---------------------------------------------------------------------------
# Nadeau-Bengio corrected paired t-test
# ---------------------------------------------------------------------------
def nadeau_bengio_ttest(scores_a, scores_b, n_train_frac=0.8):
    """Variance-corrected paired t-test for repeated k-fold CV
    (Nadeau & Bengio 2003). n2/n1 = test/train size ratio."""
    d = np.asarray(scores_a) - np.asarray(scores_b)
    k = len(d)
    mean_d = d.mean()
    var_d = d.var(ddof=1)
    if var_d == 0:
        return (np.inf if mean_d > 0 else (-np.inf if mean_d < 0 else 0.0)), \
               (0.0 if mean_d != 0 else 1.0), mean_d
    ratio = (1 - n_train_frac) / n_train_frac
    se = np.sqrt(var_d * (1.0 / k + ratio))
    t = mean_d / se
    p = 2 * stats.t.sf(abs(t), df=k - 1)
    return t, p, mean_d


def cohens_dz(scores_a, scores_b):
    d = np.asarray(scores_a) - np.asarray(scores_b)
    sd = d.std(ddof=1)
    return d.mean() / sd if sd > 0 else np.inf * np.sign(d.mean())


def holm_correction(pvals):
    """Holm step-down adjusted p-values."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    m = len(p)
    adj = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order):
        val = min(1.0, (m - rank) * p[idx])
        running_max = max(running_max, val)
        adj[idx] = running_max
    return adj


def pairwise_vs_reference(folds, metric="roc_auc", reference="TabNet",
                          variant="tuned", n_train_frac=0.8):
    """TabNet (reference) vs each baseline, per dataset, NB-corrected + Holm."""
    rows = []
    for ds in folds.dataset.unique():
        sub = folds[(folds.dataset == ds) & (folds.variant == variant)]
        piv = sub.pivot_table(index="fold_id", columns="model", values=metric)
        pvals, entries = [], []
        for m in MODELS:
            if m == reference or m not in piv.columns:
                continue
            a, b = piv[reference].values, piv[m].values
            t, p, diff = nadeau_bengio_ttest(a, b, n_train_frac)
            w_p = stats.wilcoxon(a, b).pvalue if not np.allclose(a, b) else 1.0
            entries.append({
                "dataset": ds, "comparison": f"{reference} vs {m}",
                "mean_diff": diff, "t_nb": t, "p_nb": p,
                "p_wilcoxon": w_p, "dz": cohens_dz(a, b)})
            pvals.append(p)
        adj = holm_correction(pvals)
        for e, a_p in zip(entries, adj):
            e["p_nb_holm"] = a_p
            rows.append(e)
    return pd.DataFrame(rows)


def tuned_vs_default(folds, metric="roc_auc", n_train_frac=0.8):
    """Per model+dataset: benefit of Bayesian optimization over defaults."""
    rows = []
    for ds in folds.dataset.unique():
        for m in folds.model.unique():
            sub = folds[(folds.dataset == ds) & (folds.model == m)]
            piv = sub.pivot_table(index="fold_id", columns="variant", values=metric)
            if "tuned" not in piv or "default" not in piv:
                continue
            t, p, diff = nadeau_bengio_ttest(piv["tuned"].values,
                                             piv["default"].values, n_train_frac)
            rows.append({"dataset": ds, "model": m,
                         "auc_tuned": piv["tuned"].mean(),
                         "auc_default": piv["default"].mean(),
                         "delta": diff, "t_nb": t, "p_nb": p})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Friedman / ranks across datasets
# ---------------------------------------------------------------------------
def friedman_ranks(folds, metric="roc_auc", variant="tuned"):
    """Mean metric per model per dataset -> Friedman test + average ranks."""
    means = (folds[folds.variant == variant]
             .groupby(["dataset", "model"])[metric].mean().unstack())
    means = means[[m for m in MODELS if m in means.columns]]
    ranks = means.rank(axis=1, ascending=False)
    fr_stat, fr_p = stats.friedmanchisquare(
        *[means[m].values for m in means.columns])
    k = means.shape[1]
    n = means.shape[0]
    # Nemenyi critical difference at alpha=.05
    Q_ALPHA = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
               7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164}
    cd = Q_ALPHA[k] * np.sqrt(k * (k + 1) / (6.0 * n))
    return means, ranks.mean(), fr_stat, fr_p, cd


# ---------------------------------------------------------------------------
# Calibration
# ---------------------------------------------------------------------------
def ece(y_true, proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(proba, bins) - 1, 0, n_bins - 1)
    total = len(y_true)
    err = 0.0
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0:
            continue
        err += mask.sum() / total * abs(y_true[mask].mean() - proba[mask].mean())
    return err


def calibration_table(oof, n_bins=10):
    rows = []
    for (ds, m), g in oof.groupby(["dataset", "model"]):
        y, p = g.y_true.values, g.proba.values
        rows.append({"dataset": ds, "model": m,
                     "brier": np.mean((p - y) ** 2),
                     "ece": ece(y, p, n_bins),
                     "oof_auc": stats.rankdata(p)[y == 1].mean()})  # placeholder
    from sklearn.metrics import roc_auc_score
    for r in rows:
        g = oof[(oof.dataset == r["dataset"]) & (oof.model == r["model"])]
        r["oof_auc"] = roc_auc_score(g.y_true, g.proba)
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Decision curve analysis
# ---------------------------------------------------------------------------
def net_benefit(y_true, proba, thresholds):
    n = len(y_true)
    prevalence = y_true.mean()
    out = []
    for pt in thresholds:
        pred = proba >= pt
        tp = np.sum(pred & (y_true == 1))
        fp = np.sum(pred & (y_true == 0))
        nb = tp / n - fp / n * (pt / (1 - pt))
        nb_all = prevalence - (1 - prevalence) * (pt / (1 - pt))
        out.append({"threshold": pt, "net_benefit": nb, "treat_all": nb_all,
                    "treat_none": 0.0})
    return pd.DataFrame(out)


# ---------------------------------------------------------------------------
# Operating points
# ---------------------------------------------------------------------------
def sens_at_spec(y_true, proba, target_spec=0.95):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y_true, proba)
    ok = (1 - fpr) >= target_spec
    return tpr[ok].max() if ok.any() else 0.0


def spec_at_sens(y_true, proba, target_sens=0.95):
    from sklearn.metrics import roc_curve
    fpr, tpr, thr = roc_curve(y_true, proba)
    ok = tpr >= target_sens
    return (1 - fpr[ok]).max() if ok.any() else 0.0


def operating_points(oof):
    rows = []
    for (ds, m), g in oof.groupby(["dataset", "model"]):
        y, p = g.y_true.values, g.proba.values
        rows.append({"dataset": ds, "model": m,
                     "sens_at_95spec": sens_at_spec(y, p, 0.95),
                     "spec_at_95sens": spec_at_sens(y, p, 0.95)})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# TabNet attention stability
# ---------------------------------------------------------------------------
def attention_stability(dataset):
    with open(os.path.join(RES, f"{dataset}_tabnet_importances.json")) as f:
        recs = json.load(f)
    M = np.array([r["importances"] for r in recs])  # folds x features
    names = recs[0]["feature_names"]
    n = M.shape[0]
    cors = []
    for i in range(n):
        for j in range(i + 1, n):
            cors.append(stats.spearmanr(M[i], M[j]).correlation)
    return M, names, np.array(cors)


def hyperparameter_importance(dataset, model="TabNet"):
    """Surrogate (random forest / fANOVA-style) importance of hyperparameters,
    pooled over all outer-fold Optuna studies for one dataset."""
    from sklearn.ensemble import RandomForestRegressor
    with open(os.path.join(RES, f"{dataset}_tuning.json")) as f:
        recs = json.load(f)
    rows, vals = [], []
    for r in recs:
        if r["model"] != model:
            continue
        for t in r["trials"]:
            if t["value"] is None or t["value"] <= 0:
                continue
            rows.append(t["params"])
            vals.append(t["value"])
    X = pd.DataFrame(rows)
    for c in X.columns:
        if X[c].dtype == object:
            X[c] = pd.factorize(X[c])[0]
    rf = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=1)
    rf.fit(X.fillna(-1), vals)
    return pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

print('analysis loaded')

In [ ]:
"""OPTUNE v2 — publication figures (matplotlib, journal style)."""
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# --- palette (validated categorical order; light surface) ---
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK2 = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASE = "#c3c2b7"
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
MODELS = ["TabNet", "LogisticRegression", "SVM", "RandomForest",
          "XGBoost", "LightGBM", "CatBoost", "MLP"]
MODEL_COLOR = dict(zip(MODELS, SLOTS))
MODEL_LABEL = {"TabNet": "TabNet", "LogisticRegression": "LR", "SVM": "SVM",
               "RandomForest": "RF", "XGBoost": "XGBoost",
               "LightGBM": "LightGBM", "CatBoost": "CatBoost", "MLP": "MLP"}
DS_LABEL = {"wdbc": "WDBC (n=569)", "coimbra": "Coimbra (n=116)",
            "mammographic": "Mammographic (n=961)", "cervical": "Cervical (n=858)"}
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": BASE, "axes.linewidth": 0.8,
    "axes.labelcolor": INK, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False,
})

FIG = os.path.join('.', "figures")
os.makedirs(FIG, exist_ok=True)


def _ci95(x):
    x = np.asarray(x)
    se = stats.sem(x)
    return se * stats.t.ppf(0.975, len(x) - 1)


def fig_forest_auc(folds, metric="roc_auc", fname="fig2_forest_auc.png"):
    """Mean ± 95% CI per model per dataset (tuned), forest style."""
    datasets = [d for d in ["wdbc", "coimbra", "mammographic", "cervical"]
                if d in folds.dataset.unique()]
    fig, axes = plt.subplots(1, len(datasets), figsize=(3.1 * len(datasets), 3.4),
                             sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets):
        sub = folds[(folds.dataset == ds) & (folds.variant == "tuned")]
        ypos = np.arange(len(MODELS))[::-1]
        for i, m in enumerate(MODELS):
            v = sub[sub.model == m][metric].values
            mu, h = v.mean(), _ci95(v)
            ax.errorbar(mu, ypos[i], xerr=h, fmt="o", color=MODEL_COLOR[m],
                        ecolor=MODEL_COLOR[m], ms=5, capsize=2.5, lw=1.6)
            ax.text(0.005 + ax.get_xlim()[0], ypos[i], "", fontsize=7)
        ax.set_yticks(ypos)
        ax.set_yticklabels([MODEL_LABEL[m] for m in MODELS], fontsize=8.5)
        ax.set_title(DS_LABEL[ds], fontsize=9.5, pad=6)
        ax.set_xlabel("ROC-AUC (outer folds)", fontsize=8.5)
        ax.grid(axis="x", alpha=.8)
        ax.grid(axis="y", visible=False)
    fig.suptitle("Nested-CV performance: mean ROC-AUC with 95% CI (10 outer folds)",
                 fontsize=10.5, y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_tuned_vs_default(tv, fname="fig3_tuned_vs_default.png"):
    """Dumbbell: default -> tuned AUC per model, one panel per dataset."""
    datasets = [d for d in ["wdbc", "coimbra", "mammographic", "cervical"]
                if d in tv.dataset.unique()]
    fig, axes = plt.subplots(1, len(datasets), figsize=(3.1 * len(datasets), 3.4),
                             sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets):
        sub = tv[tv.dataset == ds].set_index("model").reindex(MODELS)
        ypos = np.arange(len(MODELS))[::-1]
        for i, m in enumerate(MODELS):
            d0, d1 = sub.loc[m, "auc_default"], sub.loc[m, "auc_tuned"]
            c = MODEL_COLOR[m]
            ax.plot([d0, d1], [ypos[i], ypos[i]], color=c, lw=1.4, alpha=.65,
                    zorder=1)
            ax.scatter([d0], [ypos[i]], s=26, facecolor=SURFACE, edgecolor=c,
                       lw=1.4, zorder=2)
            ax.scatter([d1], [ypos[i]], s=30, color=c, zorder=3)
        ax.set_yticks(ypos)
        ax.set_yticklabels([MODEL_LABEL[m] for m in MODELS], fontsize=8.5)
        ax.set_title(DS_LABEL[ds], fontsize=9.5, pad=6)
        ax.set_xlabel("ROC-AUC", fontsize=8.5)
        ax.grid(axis="y", visible=False)
    handles = [plt.Line2D([], [], marker="o", ls="", mfc=SURFACE, mec=INK2,
                          label="default"),
               plt.Line2D([], [], marker="o", ls="", color=INK2, label="TPE-tuned")]
    fig.legend(handles=handles, loc="upper right", fontsize=8.5,
               bbox_to_anchor=(0.995, 1.06), ncol=2)
    fig.suptitle("Benefit of Bayesian optimization: default vs tuned (mean over folds)",
                 fontsize=10.5, y=1.04, x=0.42)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_cd_diagram(avg_ranks, cd, fname="fig4_cd_diagram.png"):
    """Critical-difference diagram (Demsar style)."""
    ranks = avg_ranks.sort_values()
    k = len(ranks)
    fig, ax = plt.subplots(figsize=(7.0, 2.6))
    lo, hi = 1, k
    ax.set_xlim(lo - 0.3, hi + 0.3)
    ax.set_ylim(0, 3.6)
    ax.axis("off")
    # axis line
    ax.plot([lo, hi], [3.0, 3.0], color=INK, lw=1.2)
    for t in range(lo, hi + 1):
        ax.plot([t, t], [3.0, 3.06], color=INK, lw=1.0)
        ax.text(t, 3.14, str(t), ha="center", fontsize=8.5, color=INK2)
    # CD bar
    ax.plot([lo, lo + cd], [3.42, 3.42], color=INK, lw=2.2)
    ax.text(lo + cd / 2, 3.50, f"CD = {cd:.2f}", ha="center", fontsize=8.5,
            color=INK)
    # model lines
    items = list(ranks.items())
    left = items[: (k + 1) // 2]
    right = items[(k + 1) // 2:]
    for i, (m, r) in enumerate(left):
        y = 2.55 - i * 0.55
        ax.plot([r, r], [3.0, y], color=MODEL_COLOR[m], lw=1.3)
        ax.plot([r, lo - 0.25], [y, y], color=MODEL_COLOR[m], lw=1.3)
        ax.text(lo - 0.28, y, f"{MODEL_LABEL[m]} ({r:.2f})", ha="right",
                va="center", fontsize=8.8, color=INK)
    for i, (m, r) in enumerate(right):
        y = 2.55 - (len(right) - 1 - i) * 0.55
        ax.plot([r, r], [3.0, y], color=MODEL_COLOR[m], lw=1.3)
        ax.plot([r, hi + 0.25], [y, y], color=MODEL_COLOR[m], lw=1.3)
        ax.text(hi + 0.28, y, f"({r:.2f}) {MODEL_LABEL[m]}", ha="left",
                va="center", fontsize=8.8, color=INK)
    ax.set_title("Average ranks across datasets (ROC-AUC), Nemenyi CD at α=0.05",
                 fontsize=10, color=INK, pad=14)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_calibration(oof, datasets=None, models_to_show=None,
                    fname="fig5_calibration.png", n_bins=8):
    """Reliability curves from out-of-fold predictions."""
    from sklearn.calibration import calibration_curve
    datasets = datasets or [d for d in ["wdbc", "coimbra", "mammographic", "cervical"]
                            if d in oof.dataset.unique()]
    models_to_show = models_to_show or ["TabNet", "LogisticRegression",
                                        "LightGBM", "SVM"]
    fig, axes = plt.subplots(1, len(datasets), figsize=(3.0 * len(datasets), 3.2),
                             sharey=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets):
        ax.plot([0, 1], [0, 1], color=BASE, lw=1.0, ls="--", zorder=1)
        for m in models_to_show:
            g = oof[(oof.dataset == ds) & (oof.model == m)]
            if g.empty:
                continue
            frac, mean_p = calibration_curve(g.y_true, g.proba, n_bins=n_bins,
                                             strategy="quantile")
            ax.plot(mean_p, frac, marker="o", ms=3.5, lw=1.6,
                    color=MODEL_COLOR[m], label=MODEL_LABEL[m])
        ax.set_title(DS_LABEL[ds], fontsize=9.5)
        ax.set_xlabel("Predicted probability", fontsize=8.5)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    axes[0].set_ylabel("Observed frequency", fontsize=8.5)
    axes[0].legend(fontsize=8, loc="upper left")
    fig.suptitle("Calibration of out-of-fold predictions (quantile bins)",
                 fontsize=10.5, y=1.03)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_decision_curves(oof, net_benefit_fn, datasets=None,
                        models_to_show=None, fname="fig6_dca.png"):
    datasets = datasets or [d for d in ["wdbc", "coimbra", "mammographic", "cervical"]
                            if d in oof.dataset.unique()]
    models_to_show = models_to_show or ["TabNet", "LogisticRegression", "LightGBM"]
    fig, axes = plt.subplots(1, len(datasets), figsize=(3.0 * len(datasets), 3.2))
    if len(datasets) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets):
        sub = oof[oof.dataset == ds]
        prev = sub[sub.model == models_to_show[0]].y_true.mean()
        thr = np.linspace(0.01, min(0.6, prev * 4), 60)
        first = True
        for m in models_to_show:
            g = sub[sub.model == m]
            if g.empty:
                continue
            nb = net_benefit_fn(g.y_true.values, g.proba.values, thr)
            if first:
                ax.plot(nb.threshold, nb.treat_all, color=MUTED, lw=1.2, ls="--",
                        label="treat all")
                ax.axhline(0, color=BASE, lw=1.0, label="treat none")
                first = False
            ax.plot(nb.threshold, nb.net_benefit, lw=1.7, color=MODEL_COLOR[m],
                    label=MODEL_LABEL[m])
        ax.set_ylim(bottom=-0.02)
        ax.set_title(DS_LABEL[ds], fontsize=9.5)
        ax.set_xlabel("Threshold probability", fontsize=8.5)
    axes[0].set_ylabel("Net benefit", fontsize=8.5)
    axes[0].legend(fontsize=7.5, loc="best")
    fig.suptitle("Decision-curve analysis (out-of-fold predictions)",
                 fontsize=10.5, y=1.03)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_importance_stability(M, names, cors, top_k=12,
                             fname="fig7_attention_stability.png"):
    """Boxplot of TabNet per-fold importances for top features + fold-pair
    Spearman distribution."""
    order = np.argsort(M.mean(axis=0))[::-1][:top_k]
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.6),
                             gridspec_kw={"width_ratios": [2.4, 1]})
    ax = axes[0]
    data = [M[:, i] for i in order][::-1]
    labels = [names[i] for i in order][::-1]
    bp = ax.boxplot(data, vert=False, patch_artist=True, widths=0.55,
                    medianprops={"color": INK, "lw": 1.2},
                    flierprops={"marker": "o", "ms": 3, "mfc": MUTED,
                                "mec": "none"})
    for patch in bp["boxes"]:
        patch.set_facecolor(SEQ_BLUE[1])
        patch.set_edgecolor(SEQ_BLUE[4])
        patch.set_linewidth(1.0)
    for el in ["whiskers", "caps"]:
        for line in bp[el]:
            line.set_color(SEQ_BLUE[4])
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("TabNet attention importance", fontsize=8.5)
    ax.set_title("Per-fold global importances (top features)", fontsize=9.5)
    ax.grid(axis="y", visible=False)

    ax = axes[1]
    ax.hist(cors, bins=12, color=SEQ_BLUE[3], edgecolor=SURFACE)
    ax.axvline(np.median(cors), color=INK, lw=1.3)
    ax.text(np.median(cors), ax.get_ylim()[1] * 0.95,
            f" median ρ = {np.median(cors):.2f}", fontsize=8.5, color=INK,
            va="top")
    ax.set_xlabel("Fold-pair Spearman ρ", fontsize=8.5)
    ax.set_ylabel("Count", fontsize=8.5)
    ax.set_title("Attribution stability across folds", fontsize=9.5)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_attribution_agreement(imp_df, fname="fig8_attribution_agreement.png"):
    """imp_df: DataFrame with columns [feature, tabnet, shap_xgb, permutation],
    normalized. Scatter + Kendall tau annotations."""
    pairs = [("tabnet", "shap_xgb", "TabNet attention vs XGBoost |SHAP|"),
             ("tabnet", "permutation", "TabNet attention vs permutation")]
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.4))
    for ax, (a, b, title) in zip(axes, pairs):
        x, y = imp_df[a], imp_df[b]
        tau = stats.kendalltau(x, y).correlation
        rho = stats.spearmanr(x, y).correlation
        ax.scatter(x, y, s=26, color=SLOTS[0], alpha=0.85, edgecolor=SURFACE,
                   lw=0.6)
        top = imp_df.nlargest(4, a)
        for _, r in top.iterrows():
            ax.annotate(r["feature"], (r[a], r[b]), fontsize=7, color=INK2,
                        xytext=(4, 3), textcoords="offset points")
        ax.set_title(f"{title}\nKendall τ = {tau:.2f}, Spearman ρ = {rho:.2f}",
                     fontsize=9)
        ax.set_xlabel("TabNet importance", fontsize=8.5)
        ax.set_ylabel("Reference importance", fontsize=8.5)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_instance_masks(masks, feature_names, y_true, fname="fig9_instance_masks.png",
                       max_instances=40, top_k=15):
    """Instance-level aggregated attention masks (test fold), sorted by class."""
    order_feat = np.argsort(masks.mean(axis=0))[::-1][:top_k]
    order_inst = np.argsort(y_true)
    Msub = masks[order_inst][:, order_feat]
    ysub = np.array(y_true)[order_inst]
    if len(ysub) > max_instances:
        keep = np.linspace(0, len(ysub) - 1, max_instances).astype(int)
        Msub, ysub = Msub[keep], ysub[keep]
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("seqblue", ["#ffffff"] + SEQ_BLUE)
    fig, ax = plt.subplots(figsize=(7.6, 4.2))
    im = ax.imshow(Msub.T, aspect="auto", cmap=cmap)
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([feature_names[i] for i in order_feat], fontsize=7.5)
    ax.set_xlabel("Test-fold patients (sorted: benign → malignant)", fontsize=8.5)
    # class divider
    split = np.searchsorted(ysub, 1)
    ax.axvline(split - 0.5, color=INK, lw=1.2)
    ax.text(split / 2, -1.2, "benign", ha="center", fontsize=8, color=INK2)
    ax.text((split + len(ysub)) / 2, -1.2, "malignant", ha="center", fontsize=8,
            color=INK2)
    cb = fig.colorbar(im, ax=ax, shrink=0.85, pad=0.01)
    cb.set_label("Aggregated attention mask", fontsize=8)
    cb.outline.set_visible(False)
    ax.set_title("Instance-level TabNet attention (WDBC held-out fold)",
                 fontsize=10, pad=10)
    ax.grid(visible=False)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)


def fig_hp_importance(series_by_ds, fname="fig10_hp_importance.png"):
    """Bar panels of surrogate hyperparameter importance per dataset (TabNet)."""
    datasets = list(series_by_ds.keys())
    fig, axes = plt.subplots(1, len(datasets), figsize=(2.9 * len(datasets), 3.0),
                             sharey=False)
    if len(datasets) == 1:
        axes = [axes]
    for ax, ds in zip(axes, datasets):
        s = series_by_ds[ds]
        ax.barh(range(len(s))[::-1], s.values, color=SEQ_BLUE[3],
                edgecolor=SURFACE, height=0.6)
        ax.set_yticks(range(len(s))[::-1])
        ax.set_yticklabels(s.index, fontsize=8)
        ax.set_title(DS_LABEL[ds], fontsize=9.5)
        ax.set_xlabel("Surrogate importance", fontsize=8.5)
        ax.grid(axis="y", visible=False)
    fig.suptitle("TabNet hyperparameter importance (RF surrogate on pooled TPE trials)",
                 fontsize=10.5, y=1.04)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, fname), dpi=300, bbox_inches="tight")
    plt.close(fig)

print('figures loaded')

### Running the benchmark

The full benchmark (~2 h on 2 CPU cores) writes per-fold metrics, out-of-fold predictions, tuning
traces, and TabNet attention importances to `results/`. The cell below skips execution when
results already exist, so this notebook re-runs analysis instantly on cached results.

In [ ]:
import os
SENTINEL = "results/wdbc_fold_metrics.csv"
RUN_FULL = not os.path.exists(SENTINEL)
DATA = [("wdbc", "data/wdbc.csv"), ("coimbra", "data/coimbra.csv"),
        ("mammographic", "data/mammographic.csv"), ("cervical", "data/cervical.csv")]
if RUN_FULL:
    missing = [p for _, p in DATA if not os.path.exists(p)]
    if missing:
        print("cannot run benchmark - missing input data:")
        for p in missing:
            print("  ", p)
        print("place the four dataset CSVs under data/ then re-run this cell")
    else:
        for ds, path in DATA:
            run_benchmark(ds, path, "results")
        print("benchmark complete - results written to results/")
else:
    print("cached results found - skipping (delete results/ to re-run)")


## 4. Results — discrimination performance

Mean ROC-AUC over the 10 outer folds, with 95% CIs. Note how different these honest nested-CV
numbers are from v1's selection-biased holdout numbers: on WDBC, v1 reported a validation AUC of
0.9996 for TabNet; the unbiased estimate for the strongest models is ≈0.99, and **no model separates
from the others on WDBC** — the dataset is saturated, which is precisely why a multi-dataset design
is necessary.

In [ ]:
DATASETS = ["wdbc", "coimbra", "mammographic", "cervical"]

def load_merged(suffix):
    frames = []
    for d in DATASETS:
        path = f"results/{d}_{suffix}.csv"
        if not os.path.exists(path):
            print(f"missing: {path}")
            continue
        frames.append(pd.read_csv(path))
    if not frames:
        raise FileNotFoundError(
            f"no *_{suffix}.csv under results/ - run the benchmark cell in Section 3 first")
    return pd.concat(frames, ignore_index=True)

folds = load_merged("fold_metrics")
oof   = load_merged("oof_predictions")


In [ ]:
fig_forest_auc(folds)
from IPython.display import Image, display
display(Image("figures/fig2_forest_auc.png", width=980))

### Statistical comparison (TabNet vs each baseline)

Nadeau–Bengio corrected paired t-tests on outer-fold ROC-AUC, Holm-adjusted within each dataset.
Positive `mean_diff` favours TabNet.

In [ ]:
t3 = pairwise_vs_reference(folds, metric="roc_auc")
t3[["dataset", "comparison", "mean_diff", "t_nb", "p_nb", "p_nb_holm", "dz",
    "p_wilcoxon"]].round(4)

In [ ]:
means, avg_ranks, fr_stat, fr_p, cd = friedman_ranks(folds)
print(f"Friedman chi-square = {fr_stat:.2f}, p = {fr_p:.4f}; Nemenyi CD (alpha=.05) = {cd:.2f}")
fig_cd_diagram(avg_ranks, cd)
display(Image("figures/fig4_cd_diagram.png", width=760))

## 5. The value of Bayesian optimization (RQ3 revisited)

Because tuned and default variants of every model were evaluated on identical outer folds, the
tuned-default difference is a paired, unbiased estimate of what TPE search contributes.

In [ ]:
tv = tuned_vs_default(folds)
tv.round(4).sort_values(["dataset", "delta"], ascending=[True, False])

In [ ]:
fig_tuned_vs_default(tv)
display(Image("figures/fig3_tuned_vs_default.png", width=980))

In [ ]:
import json
from IPython.display import Image, display
series = {}
for ds in DATASETS:
    path = f"results/{ds}_tuning.json"
    if not os.path.exists(path):
        print(f"missing: {path} - skipping"); continue
    with open(path) as f:
        recs = json.load(f)
    from sklearn.ensemble import RandomForestRegressor
    rows, vals = [], []
    for r in recs:
        if r["model"] != "TabNet":
            continue
        for t in r["trials"]:
            if t["value"] and t["value"] > 0:
                rows.append(t["params"]); vals.append(t["value"])
    X = pd.DataFrame(rows)
    for c in X.columns:
        if X[c].dtype == object:
            X[c] = pd.factorize(X[c])[0]
    rf = RandomForestRegressor(n_estimators=500, random_state=0).fit(X.fillna(-1), vals)
    series[ds] = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
if series:
    fig_hp_importance(series)
    display(Image("figures/fig10_hp_importance.png", width=980))
else:
    print("no tuning results found - run the benchmark cell in Section 3 first")


## 6. Clinical utility — calibration, decision curves, operating points

Discrimination alone is not clinical utility. Out-of-fold predicted probabilities (every patient
scored by a model that never saw them) support honest calibration and net-benefit analysis.

In [ ]:
fig_calibration(oof)
display(Image("figures/fig5_calibration.png", width=980))

In [ ]:
cal = calibration_table(oof); op = operating_points(oof)
cal.merge(op, on=["dataset", "model"]).round(4).sort_values(["dataset", "brier"])

In [ ]:
fig_decision_curves(oof, net_benefit, models_to_show=["TabNet", "LogisticRegression", "LightGBM"])
display(Image("figures/fig6_dca.png", width=980))

## 7. Interpretability with rigor

v1 showed a single feature-importance bar chart from one trained model. A claim that "TabNet's
attention aligns with clinical knowledge" requires showing that attributions are (a) **stable**
across resampling and (b) **consistent** with independent attribution methods.

In [ ]:
import numpy as np, json
from scipy import stats as st
from IPython.display import Image, display
path = "results/wdbc_tabnet_importances.json"
if not os.path.exists(path):
    print(f"missing: {path} - run the benchmark cell in Section 3 first")
else:
    with open(path) as f:
        recs = json.load(f)
    M = np.array([r["importances"] for r in recs]); names = recs[0]["feature_names"]
    cors = [st.spearmanr(M[i], M[j]).correlation
            for i in range(len(M)) for j in range(i + 1, len(M))]
    fig_importance_stability(M, names, np.array(cors))
    display(Image("figures/fig7_attention_stability.png", width=980))


In [ ]:
import numpy as np, json
from IPython.display import Image, display
need = ["results/wdbc_tabnet_importances.json",
        "results/wdbc_tabnet_permutation.csv",
        "results/wdbc_xgb_shap.csv"]
missing = [p for p in need if not os.path.exists(p)]
if missing:
    for p in missing:
        print(f"missing: {p}")
    print("run the benchmark cell in Section 3 first (writes permutation + SHAP artifacts)")
else:
    with open(need[0]) as f:
        _recs = json.load(f)
    M = np.array([r["importances"] for r in _recs]); names = _recs[0]["feature_names"]
    perm = pd.read_csv(need[1])
    shp = pd.read_csv(need[2])
    feat_cols = [c for c in perm.columns if c not in ("repeat", "fold")]
    imp_df = pd.DataFrame({
        "feature": feat_cols,
        "tabnet": pd.Series(M.mean(axis=0), index=names).reindex(feat_cols).values,
        "shap_xgb": (shp[feat_cols].mean() / shp[feat_cols].mean().sum()).values,
        "permutation": (perm[feat_cols].mean().clip(lower=0)
                        / perm[feat_cols].mean().clip(lower=0).sum()).values})
    fig_attribution_agreement(imp_df)
    display(Image("figures/fig8_attribution_agreement.png", width=880))


In [ ]:
import numpy as np
from IPython.display import Image, display
path = "results/wdbc_instance_masks.npz"
if not os.path.exists(path):
    print(f"missing: {path} - run the benchmark cell in Section 3 first (writes TabNet masks)")
else:
    z = np.load(path, allow_pickle=True)
    masks = z["masks"]; masks = masks / masks.sum(axis=1, keepdims=True).clip(min=1e-9)
    fig_instance_masks(masks, list(z["feature_names"]), z["y_true"])
    display(Image("figures/fig9_instance_masks.png", width=880))


## 8. Key findings

1. **Honest performance estimates.** Under leakage-free nested CV, no single model dominates:
   WDBC is saturated (seven of eight models above 0.99 AUC within one SD of each other), while harder
   tasks (Coimbra, cervical biopsy, AUC ceilings ~0.82 and ~0.66) expose real differences. Claims
   built on a single saturated benchmark do not generalize - the multi-dataset design is the
   substance of the contribution.
2. **TabNet does not earn its complexity on these datasets.** With equal tuning budgets it has
   the worst average rank (7.75/8 across datasets), although no pairwise difference survives the
   conservative Nadeau-Bengio correction on 10 folds. This reframes the paper from "TabNet wins"
   to "when does attention-based tabular DL earn its complexity?" - consistent with, and adding
   cancer-specific evidence to, Grinsztajn et al. (2022) and Shwartz-Ziv & Armon (2022).
3. **Bayesian optimization helps most where variance is high** (small n, weak signal) and helps
   deep/complex models more than linear ones (Section 5).
4. **Calibration and net benefit differ across models with similar AUC** — the clinically relevant
   distinction v1 could not see.
5. **TabNet's attention explanations are less trustworthy than v1 claimed.** Averaged over folds
   the importance profile recovers the classical extreme-morphology ("worst") features and agrees
   moderately with permutation importance (Kendall tau ~ 0.55), but per-fold attention rankings are
   unstable (median fold-pair Spearman rho ~ 0.14) and agreement with TreeSHAP is weak (tau ~ 0.29).
   Attention masks should not be presented as reliable per-patient explanations without a
   stabilization strategy - an important, publishable negative finding.

## 9. Limitations and reproducibility

- All datasets are retrospective public benchmarks; external, temporally separated validation
  remains necessary before any clinical claim.
- The TPE budget (15 trials × 3 inner folds) is deliberately modest and equal across models;
  larger budgets could change absolute numbers but not the fairness of the comparison.
- TabPFN could not be included in this run (pretrained weights unavailable in the sandbox);
  the framework supports it via `eval_model_on_split`.
- Every random seed is fixed and derived from `(repeat, fold)`; per-fold tuning traces,
  out-of-fold predictions, and all tables/figures are written to `results/` and `figures/`.
